# Generate the Nestle 1904 Greek New Testament documentation
This notebook builds the complete **N1904** example using the current context-first Doc4CF workflow. Run the cells in order from a Doc4CF checkout with a Python 3.13+ kernel, Git, and network access for installation/source download.

Source: [CenterBLC/N1904](https://github.com/CenterBLC/N1904), TF version **1.0.0**, pinned revision `3f53a5a9075cbca5c3001a000a7b004cdee251b6`. The notebook checks out only that revision in its own workspace. Generated files and caches stay in ignored local directories until the optional publication step.

LLM enrichment is off by default. Set `USE_LLM = True` to create and review dataset/family context artifacts before generating context-aware feature descriptions. It uses **nvidia/nemotron-3.5-super-vl:free** by default and includes the N1904 editorial notes as background during feature generation. New model responses may differ from checked-in prose. All measured corpus pages remain based on the pinned dataset.

## 1. Locate and install Doc4CF
Select the Python 3.13+ kernel before running this cell. The editable install uses the code in this checkout; restart the kernel if your notebook environment asks you to.

In [ ]:
from pathlib import Path
import sys
import subprocess
import shutil
import os
import json

assert sys.version_info >= (3, 13), "Select a Python 3.13+ kernel."
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "doc4cf").is_dir()), None)
assert ROOT is not None, "Open this notebook inside the Doc4CF checkout."
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT)], check=True)
assert shutil.which("git"), "Install Git and restart the kernel."

## 2. Fetch the pinned corpus
Only the base TF version is used, without optional annotation modules. A dedicated clone under `.testdata/example-sources` keeps source data separate from generated HTML. Existing clones are reused only if clean; the revision check prevents silent changes to the source.

In [ ]:
NAME = 'N1904'
REVISION = '3f53a5a9075cbca5c3001a000a7b004cdee251b6'
VERSION = '1.0.0'
SOURCE = ROOT / ".testdata" / "example-sources" / NAME
SOURCE.parent.mkdir(parents=True, exist_ok=True)
if not SOURCE.exists():
    subprocess.run(["git", "clone", "--filter=blob:none",
                    "https://github.com/CenterBLC/N1904.git", str(SOURCE)], check=True)
#assert not subprocess.check_output(["git", "-C", str(SOURCE), "status", "--porcelain"], text=True).strip(), "Source checkout has local changes."
subprocess.run(["git", "-C", str(SOURCE), "fetch", "origin", REVISION], check=True)
subprocess.run(["git", "-C", str(SOURCE), "checkout", "--detach", REVISION], check=True)
assert subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip() == REVISION
CORPUS = SOURCE / "tf" / VERSION
assert len(list(CORPUS.glob("*.tf"))) == 61
assert (CORPUS / "otype.tf").is_file() and (CORPUS / "oslots.tf").is_file()

## 3. Configure context analysis and enrichment

`USE_LLM = False` builds the deterministic site without model calls. Set it to `True` to run the context-first workflow. Set `TARGETED_EVIDENCE = True` only if you want a planning request to retrieve rare values or examples before each feature description; this adds model requests. The context pages and review report are enabled for enriched builds.

The OpenRouter key is read from `OPENROUTER_API_KEY` or entered through a hidden prompt. It is never included in a command or notebook output. The editorial notes path is configured below. The N1904 editorial notes accompany feature evidence during feature generation. They are editorial background, not measured corpus facts.

Review the dataset, family, and relationship artifacts before generating feature descriptions or publishing the site.

In [ ]:
USE_LLM = False
TARGETED_EVIDENCE = False
CONTEXT_PAGES = True
PROVIDER = "openrouter"
DATASET_MODEL = 'nvidia/nemotron-3.5-super-vl:free'
FAMILY_MODEL = 'nvidia/nemotron-3.5-super-vl:free'
FEATURE_MODEL = 'nvidia/nemotron-3.5-super-vl:free'
CONTEXT = ROOT / "notebooks" / "contexts" / "N1904.txt"
CONTEXT_DIR = ROOT / ".testdata" / "context-artifacts" / 'N1904'
OUTPUT = ROOT / ".testdata" / "example-builds" / 'N1904' / ("enriched" if USE_LLM else "offline")
CACHE = ROOT / ".doc4cf-cache" / 'n1904'

if USE_LLM:
    from getpass import getpass
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")
    assert os.environ.get("OPENROUTER_API_KEY"), "An API key is required for enrichment."

## 4. Analyze the dataset and inspect context

This standalone step writes `inventory.json`, `dataset_context.json`, `feature_families.json`, and `feature_relationships.json`. Context analysis uses separate dataset and family model settings, is cached, and does not build the site. If a request is interrupted, rerun this cell after the provider is available; completed responses are reused.

In [ ]:
if USE_LLM:
    analysis_command = [
        sys.executable, "-m", "doc4cf", str(CORPUS),
        "--analyze-context", str(CONTEXT_DIR),
        "--provider", PROVIDER,
        "--context-model", DATASET_MODEL,
        "--family-model", FAMILY_MODEL,
        "--cache", str(CACHE),
    ]
    analysis = subprocess.run(analysis_command, cwd=ROOT, capture_output=True,
                             text=True, encoding="utf-8", errors="replace")
    print(analysis.stdout)
    print(analysis.stderr)
    if analysis.returncode:
        raise RuntimeError("Context analysis failed; review the error and rerun this cell.")
    for artifact in ("inventory.json", "dataset_context.json", "feature_families.json", "feature_relationships.json"):
        assert (CONTEXT_DIR / artifact).is_file(), artifact
    print("Context artifacts:", CONTEXT_DIR)
else:
    print("Context analysis skipped because USE_LLM is False.")

### Review the generated artifacts

Inspect the dataset interpretation, feature families, relationship statuses, evidence paths, and `needs_review` fields. These files are the reusable model context for the next step. If you revise them, retain the artifact structure and inventory hash so Doc4CF can validate them.

In [ ]:
if USE_LLM:
    for artifact in ("dataset_context.json", "feature_families.json", "feature_relationships.json"):
        print("\n###", artifact)
        print((CONTEXT_DIR / artifact).read_text(encoding="utf-8"))

## 5. Generate the site with reviewed context

The regular build consumes the context artifacts, checks their inventory hash against the pinned corpus, and writes dataset architecture/glossary, relationship, and context review pages. N1904 editorial notes, when configured, accompany feature evidence. `--targeted-evidence` is opt-in and retrieves requested values/examples locally before the final description request.

If generation stops on a provider quota or transient failure, rerun the cell. Successful context and feature responses remain cached. Review all generated language and the `context_review.json` report before publication.

In [ ]:
command = [sys.executable, "-m", "doc4cf", str(CORPUS), str(OUTPUT),
           "--project-home", "../../index.html"]
if USE_LLM:
    command += ["--provider", PROVIDER, "--model", FEATURE_MODEL,
                "--feature-model", FEATURE_MODEL,
                "--context-dir", str(CONTEXT_DIR), "--context-pages",
                "--model-attribution", "feature", "--cache", str(CACHE)]
    if TARGETED_EVIDENCE:
        command.append("--targeted-evidence")
    if CONTEXT is not None:
        CONTEXT = Path(CONTEXT)
        assert CONTEXT.is_file(), CONTEXT
        command += ["--dataset-context", str(CONTEXT)]

result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                        encoding="utf-8", errors="replace")
print("EXIT CODE:", result.returncode)
print("\n--- STDOUT ---")
print(result.stdout)
print("\n--- STDERR ---")
print(result.stderr)
if result.returncode:
    raise RuntimeError("Documentation generation failed; do not publish its partial output.")
shutil.copyfile(SOURCE / 'LICENSE.md', OUTPUT / 'DATA-LICENSE.md')
print("Generated:", OUTPUT / "index.html")

## 6. Check completeness and inspect the pages

Every TF file should have a feature page and graph node. Inspect the overview, dataset context, relationships, context review, structure, and diagram pages. Sample generated descriptions and confirm that model interpretations and uncertainty flags are appropriately grounded.

In [ ]:
features = {p.stem for p in CORPUS.glob("*.tf")}
assert {p.stem for p in (OUTPUT / "features").glob("*.html")} == features
graph = json.loads((OUTPUT / "knowledge_graph.json").read_text(encoding="utf-8"))
assert {"feature::" + f for f in features} <= graph["nodes"].keys()
required_pages = ["index.html", "features.html", "structure.html", "diagram.html", "text.html"]
if USE_LLM and CONTEXT_PAGES:
    required_pages += ["dataset.html", "relationships.html", "context-review.html", "context_review.json", "context_manifest.json"]
for page in required_pages:
    assert (OUTPUT / page).is_file(), page
for name in features:
    html = (OUTPUT / "features" / (name + ".html")).read_text(encoding="utf-8")
    assert ("Generated description" in html) == USE_LLM
print(f"Verified {len(features)} feature pages and {len(required_pages)} overview/review pages.")
print((OUTPUT / "index.html").as_uri())

## 7. Optional: replace the checked-in example

Set `PUBLISH = True` only after reviewing the generated pages and context review report. This copies the build into `examples/<corpus>/`; it makes no Git commit or push. An offline build replaces enriched feature descriptions with corpus-only pages. The Pages workflow publishes checked-in files without calling an LLM.

Inspect `git diff` before committing. Notebook outputs are cleared before sharing. Keep credentials, context caches, and downloaded corpora out of Git.

In [ ]:
PUBLISH = False
if PUBLISH:
    destination = ROOT / "examples" / NAME
    shutil.copytree(OUTPUT, destination, dirs_exist_ok=True)
    print("Updated:", destination)
else:
    print("Review build retained locally; checked-in example unchanged.")